In [1]:
from PyPDF2 import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import DashScopeEmbeddings
from langchain_community.vectorstores import FAISS
from typing import List, Tuple
import os
import pickle

DASHSCOPE_API_KEY = os.getenv('DASHSCOPE_API_KEY')
if not DASHSCOPE_API_KEY:
    raise ValueError("请设置环境变量 DASHSCOPE_API_KEY")

def extract_text_with_page_numbers(pdf) -> Tuple[str, List[Tuple[str, int]]]:
    """
    从PDF中提取文本并记录每个字符对应的页码
    
    参数:
        pdf: PDF文件对象
    
    返回:
        text: 提取的文本内容
        char_page_mapping: 每个字符对应的页码列表
    """
    text = ""
    char_page_mapping = []

    for page_number, page in enumerate(pdf.pages, start=1):
        extracted_text = page.extract_text()
        if extracted_text:
            text += extracted_text
            # 为当前页面的每个字符记录页码
            char_page_mapping.extend([page_number] * len(extracted_text))
        else:
            print(f"No text found on page {page_number}.")

    return text, char_page_mapping

def process_text_with_splitter(text: str, char_page_mapping: List[int], save_path: str = None) -> FAISS:
    """
    处理文本并创建向量存储
    
    参数:
        text: 提取的文本内容
        char_page_mapping: 每个字符对应的页码列表
        save_path: 可选，保存向量数据库的路径
    
    返回:
        knowledgeBase: 基于FAISS的向量存储对象
    """
    # 创建文本分割器，用于将长文本分割成小块
    text_splitter = RecursiveCharacterTextSplitter(
        separators=["\n\n", "\n", ".", " ", ""],
        chunk_size=1000,
        chunk_overlap=200,
        length_function=len,
    )

    # 分割文本
    chunks = text_splitter.split_text(text)
    print(f"文本被分割成 {len(chunks)} 个块。")
        
    # 创建嵌入模型
    embeddings = DashScopeEmbeddings(
        model="text-embedding-v1",
        dashscope_api_key=DASHSCOPE_API_KEY,
    )
    
    # 从文本块创建知识库
    knowledgeBase = FAISS.from_texts(chunks, embeddings)
    print("已从文本块创建知识库。")
    
    # 为每个文本块找到对应的页码信息
    page_info = {}
    current_pos = 0
    
    for chunk in chunks:
        chunk_start = current_pos
        chunk_end = current_pos + len(chunk)
        
        # 找到这个文本块中字符对应的页码
        chunk_pages = char_page_mapping[chunk_start:chunk_end]
        
        # 取页码的众数（出现最多的页码）作为该块的页码
        if chunk_pages:
            # 统计每个页码出现的次数
            page_counts = {}
            for page in chunk_pages:
                page_counts[page] = page_counts.get(page, 0) + 1
            
            # 找到出现次数最多的页码
            most_common_page = max(page_counts, key=page_counts.get)
            page_info[chunk] = most_common_page
        else:
            page_info[chunk] = 1  # 默认页码
        
        current_pos = chunk_end
    
    knowledgeBase.page_info = page_info
    print(f'页码映射完成，共 {len(page_info)} 个文本块')
    
    # 如果提供了保存路径，则保存向量数据库和页码信息
    if save_path:
        # 确保目录存在
        os.makedirs(save_path, exist_ok=True)
        
        # 保存FAISS向量数据库
        knowledgeBase.save_local(save_path)
        print(f"向量数据库已保存到: {save_path}")
        
        # 保存页码信息到同一目录
        with open(os.path.join(save_path, "page_info.pkl"), "wb") as f:
            pickle.dump(page_info, f)
        print(f"页码信息已保存到: {os.path.join(save_path, 'page_info.pkl')}")
    
    return knowledgeBase

def load_knowledge_base(load_path: str, embeddings = None) -> FAISS:
    """
    从磁盘加载向量数据库和页码信息
    
    参数:
        load_path: 向量数据库的保存路径
        embeddings: 可选，嵌入模型。如果为None，将创建一个新的DashScopeEmbeddings实例
    
    返回:
        knowledgeBase: 加载的FAISS向量数据库对象
    """
    # 如果没有提供嵌入模型，则创建一个新的
    if embeddings is None:
        embeddings = DashScopeEmbeddings(
            model="text-embedding-v1",
            dashscope_api_key=DASHSCOPE_API_KEY,
        )
    
    # 加载FAISS向量数据库，添加allow_dangerous_deserialization=True参数以允许反序列化
    knowledgeBase = FAISS.load_local(load_path, embeddings, allow_dangerous_deserialization=True)
    print(f"向量数据库已从 {load_path} 加载。")
    
    # 加载页码信息
    page_info_path = os.path.join(load_path, "page_info.pkl")
    if os.path.exists(page_info_path):
        with open(page_info_path, "rb") as f:
            page_info = pickle.load(f)
        knowledgeBase.page_info = page_info
        print("页码信息已加载。")
    else:
        print("警告: 未找到页码信息文件。")
    
    return knowledgeBase

# 读取PDF文件
pdf_reader = PdfReader('./财报_中芯国际：中芯国际2025年年度报告.pdf')
# 提取文本和页码信息
text, char_page_mapping = extract_text_with_page_numbers(pdf_reader)
text

'中芯国际集成电路制造有限公司2025年年度报告\n1/232公司代码：688981 公司简称：中芯国际\n中芯国际集成电路制造有限公司\n2025年年度报告中芯国际集成电路制造有限公司2025年年度报告\n2/232重要提示\n一、本公司董事会及董事、高级管理人员保证年度报告内容的真实性、准确性、完整性，不存在\n虚假记载、误导性陈述或重大遗漏，并承担个别和连带的法律责任。\n二、公司上市时未盈利且尚未实现盈利\n□是√否\n三、重大风险提示\n公司已在本报告中详细阐述公司在生产经营过程中可能面临的各种风险及应对措施，敬请查\n阅本报告“第四节管理层讨论与分析”之“四、风险因素”。\n四、公司全体董事出席董事会会议。\n五、安永华明会计师事务所（特殊普通合伙）为本公司出具了标准无保留意见的审计报告。\n六、公司负责人刘训峰、主管会计工作负责人吴俊峰及会计机构负责人（会计主管人员）刘晨健\n声明：保证年度报告中财务报告的真实、准确、完整。\n七、董事会决议通过的本报告期利润分配预案或公积金转增股本预案\n公司2025年度拟不进行利润分配，该议案尚需提交2026年年度股东大会审议。\n母公司存在未弥补亏损\n□适用√不适用\n八、是否存在公司治理特殊安排等重要事项\n√适用□不适用\n公司治理特殊安排情况：\n√本公司为红筹企业\n□本公司存在协议控制架构\n□本公司存在表决权差异安排\n九、前瞻性陈述的风险声明\n√适用□不适用\n本报告可能载有（除历史数据外）前瞻性陈述。该等前瞻性陈述乃根据中芯国际对未来事件\n或绩效的现行假设、期望、信念、计划、目标及预测而作出。中芯国际使用包括（但不限于)“相\n信”、“预期”、“打算”、“估计”、“预计”、“预测”、“指标”、“展望”、“继续”\n、“应该”、“或许”、“寻求”、“应当”、“计划”、“可能”、“愿景”、“目标”、“\n旨在”、“渴望”、“目的”、“预定”、“前景”和其他类似的表述，以识别前瞻性陈述。该中芯国际集成电路制造有限公司2025年年度报告\n3/232等前瞻性陈述乃反映中芯国际高级管理层根据最佳判断作出的估计，存在重大已知及未知的风险\n、不确定性以及其他可能导致中芯国际实际业绩、财务状况或经营结果与前瞻性陈述所载数据有\n重大差异的因素，包括（但不限于）与半导体行业周期及市场情况有关风险、半导体行业

In [2]:
print(f"提取的文本长度: {len(text)} 个字符。")
    
# 处理文本并创建知识库，同时保存到磁盘
save_dir = "./vector_db"
knowledgeBase = process_text_with_splitter(text, char_page_mapping, save_path=save_dir)

提取的文本长度: 206599 个字符。
文本被分割成 259 个块。
已从文本块创建知识库。
页码映射完成，共 259 个文本块
向量数据库已保存到: ./vector_db
页码信息已保存到: ./vector_db\page_info.pkl


In [4]:
from langchain_community.llms import Tongyi
llm = Tongyi(model_name="deepseek-v3", dashscope_api_key=DASHSCOPE_API_KEY) # qwen-turbo

# 设置查询问题
#query = "中芯国际2025年的营业收入、净利润和毛利率分别是多少？"
query = "公司前两大股东是谁，持股比例是多少？"
if query:
    # 执行相似度搜索，找到与查询相关的文档
    docs = knowledgeBase.similarity_search(query, k=4)

    # 构建上下文
    context = "\n\n".join([doc.page_content for doc in docs])

    # 构建提示
    prompt = f"""根据以下上下文回答问题:

{context}

问题: {query}"""

    # 直接调用 LLM
    response = llm.invoke(prompt)
    print(response)
    print("来源:")

    # 记录唯一的页码
    unique_pages = set()

    # 显示每个文档块的来源页码
    for doc in docs:
        #print('doc=',doc)
        text_content = getattr(doc, "page_content", "")
        source_page = knowledgeBase.page_info.get(
            text_content.strip(), "未知"
        )

        if source_page not in unique_pages:
            unique_pages.add(source_page)
            print(f"文本块页码: {source_page}")

根据提供的《中芯国际集成电路制造有限公司2025年年度报告》（第117–119页等）内容，需明确区分“前十名股东持股情况表”（A股+港股穿透后披露）与“香港权益披露”部分的口径。关键在于：**报告明确说明，HKSCCNOMINEES LIMITED（香港中央结算（代理人）有限公司）是名义持有人，代表大量非登记股东（主要是港股通及境外投资者），其持股不反映实际控制权，且公司已将已知的大股东（如大唐香港、鑫芯香港）从其名下剔除以避免重复计算**。

综合多处信息，特别是：

- **“截至报告期末前十名股东持股情况（不含通过转融通出借股份）”表（第118页）**  
- **“第3部分规定须向本公司披露……5%或以上已发行股份权益”表（第115页）**，该表按实益权益（beneficial ownership）口径，基于总股本8,000,408,035股计算，更具实质性和可比性。

我们提取并核实前两大实益股东（即真正拥有控制权或重大权益的股东）：

1. **中国信息通信科技集团有限公司（中国信科）**  
   - 权益性质：好仓，实益拥有人/受控制法团权益  
   - 持股总数：**1,197,513,450 股**  
     （其中：直接持有72,470,855股 + 间接通过全资子公司大唐控股（香港）投资有限公司持有1,125,042,595股）  
   - 占总股本比例：**14.97%**（注：报告原文为“14.97%”，与计算一致：1,197,513,450 ÷ 8,000,408,035 ≈ 14.97%）

2. **国家集成电路产业投资基金股份有限公司（国家集成电路基金一期）**  
   - 权益性质：好仓，实益拥有人/受控制法团权益  
   - 持股总数：**740,245,419 股**  
     （其中：直接持有357,343,396股 + 间接通过全资子公司鑫芯（香港）投资有限公司持有382,902,023股）  
   - 占总股本比例：**9.25%**（740,245,419 ÷ 8,000,408,035 ≈ 9.25%）

✅ 注意排除干扰项：
- HKSCCNOMINEES LIMITED 虽在“前十名股东表”中列第一（持股55.99%），但报告第118页脚注第3条明确指出：“公司已将大唐香港、鑫芯香港等已知大股

In [4]:
docs

[Document(id='57206d6f-4a36-4129-a360-e2dcb7bed36c', metadata={}, page_content='资产和其他长期资产支付的现金为人民币59,950.6百万元，较上年同期增加9.9%。中芯国际集成电路制造有限公司2025年年度报告\n25/232\n主营业务分析\n1.利润表及现金流量表相关科目变动分析表\n单位：千元币种：人民币\n科目 本期数 上年同期数 变动比例（%）\n营业收入 67,323,192 57,795,570 16.5\n营业成本 52,765,123 47,051,267 12.1\n毛利 14,558,069 10,744,303 35.5\n销售费用 304,795 281,549 8.3\n管理费用 3,434,995 3,835,368 -10.4\n研发费用 5,518,596 5,447,122 1.3\n财务费用 -370,874 -1,832,649 不适用\n投资收益 -56,415 1,099,723 -105.1\n营业外支出 540,723 23,138 2,236.9\n所得税费用 576,736 918,904 -37.2\n经营活动产生的现金流量净额 20,080,979 22,658,629 -11.4\n投资活动产生的现金流量净额 -42,138,175 -30,669,293 不适用\n筹资活动产生的现金流量净额 17,453,618 9,998,953 74.6\n(1)营业收入变动原因说明：主要是由于本年晶圆销量增加所致。销售晶圆的数量（折合8英寸\n标准逻辑）由上年的802.1万片增加20.9%至本年的969.7万片。平均售价（销售晶圆收入除\n以总销售晶圆数量）本年为人民币6,476元，上年为人民币6,639元。\n(2)营业成本变动原因说明：主要是由于本年晶圆销量增加、产品组合变动及折旧增加所致。\n(3)毛利变动原因说明：主要是由于本年晶圆销量增加、产能利用率上升及产品组合变动所致。\n(4)财务费用变动原因说明：主要是由于本年利息收入减少和利息支出增加所致。\n(5)投资收益变动原因说明：主要是由于上年出售一联营企业产生收益，而本年未发生所致。\n(6)营业外支出变动原因说明：主要是由于本年认列诉讼相关费用所致。\n(7)所得税费用变动